In [1]:
import os
import pandas as pd
import numpy as np
import random
import dendropy
import treeswift
from treeswift import read_tree_newick
#import matplotlib.pyplot as plt
import re
import copy

In [9]:
path = "cafe_dist"
phylip_mtrx = "results.D2star.phylip"
#phylip_mtrx = "results.D2shepp.phylip"
#phylip_mtrx = "results.CVtree.phylip"
#phylip_mtrx = "results.Cosine.phylip"

#phylip_mtrx = "results.Co-phylog.phylip"
#phylip_mtrx = "results.Eu.phylip"
#phylip_mtrx = "results.JS.phylip"
#phylip_mtrx = "results.Ma.phylip"

dist_name = phylip_mtrx.split(".")[1]

# Read phylip matrix
def read_phylip_dist(filename):
    with open(filename) as f:
        # first line = number of taxa (ignore or check consistency)
        n = int(f.readline().strip())
        labels = []
        matrix = []
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            labels.append(parts[0])         # first entry = sequence ID
            matrix.append([float(x) for x in parts[1:]])  # rest = distances

    # Create square DataFrame
    df = pd.DataFrame(matrix, index=labels, columns=labels)
    return df

# Example usage
df = read_phylip_dist(os.path.join(path, phylip_mtrx))
print(df.head())

                                   G000430525.part_NZ_AUME01000007.1  \
G000430525.part_NZ_AUME01000007.1                           0.000000   
G000424365.part_NZ_AUJT01000008.1                           0.352161   
G000336655.part_NZ_AOLZ01000076.1                           0.448177   
G000163775.part_NZ_JH815302.1                               0.329393   
G000430745.part_NZ_KE387231.1                               0.301708   

                                   G000424365.part_NZ_AUJT01000008.1  \
G000430525.part_NZ_AUME01000007.1                           0.352161   
G000424365.part_NZ_AUJT01000008.1                           0.000000   
G000336655.part_NZ_AOLZ01000076.1                           0.466374   
G000163775.part_NZ_JH815302.1                               0.458266   
G000430745.part_NZ_KE387231.1                               0.336400   

                                   G000336655.part_NZ_AOLZ01000076.1  \
G000430525.part_NZ_AUME01000007.1                           0.

In [10]:
df.head(n=5)

,G000430525.part_NZ_AUME01000007.1,G000424365.part_NZ_AUJT01000008.1,G000336655.part_NZ_AOLZ01000076.1,G000163775.part_NZ_JH815302.1,G000430745.part_NZ_KE387231.1,G001894865.part_NZ_BCXA01000020.1,G900110905.part_FOFH01000009.1,G000300115.part_NZ_JH930378.1,G000172095.part_NZ_ABID01000003.1,G000430745.part_NZ_AUMP01000016.1,...,G900097105.part_NZ_LT629973.1,G001313205.part_BBFK01000002.1,G000336655.part_NZ_AOLZ01000022.1,G001005215.part_NZ_LBNQ01000037.1,G000497755.part_NZ_AYOD01000011.1,G000519205.part_NZ_JAFB01000009.1,G000224335.part_NZ_AFXZ01000019.1,G001308105.part_NZ_CP012851.1,G000425585.part_NZ_AUDS01000008.1,G000719275.part_NZ_JOFP01000004.1
G000430525.part_NZ_AUME01000007.1,0.000000,0.352161,0.448177,0.329393,0.301708,0.402909,0.374598,0.374444,0.287954,0.317699,...,0.247781,0.344106,0.453857,0.295523,0.337603,0.372078,0.338929,0.297073,0.286769,0.389754
G000424365.part_NZ_AUJT01000008.1,0.352161,0.000000,0.466374,0.458266,0.336400,0.426815,0.400139,0.441435,0.353202,0.332600,...,0.290732,0.446449,0.469780,0.368003,0.378607,0.406565,0.415452,0.298610,0.275142,0.419159
G000336655.part_NZ_AOLZ01000076.1,0.448177,0.466374,0.000000,0.541569,0.455113,0.163450,0.487712,0.536074,0.425270,0.445568,...,0.356746,0.457994,0.029615,0.452411,0.215755,0.191424,0.519376,0.538388,0.446742,0.188228
G000163775.part_NZ_JH815302.1,0.329393,0.458266,0.541569,0.000000,0.312728,0.497701,0.269718,0.315861,0.297955,0.320349,...,0.328771,0.287472,0.541356,0.327808,0.436320,0.460054,0.265488,0.281080,0.287741,0.466396
G000430745.part_NZ_KE387231.1,0.301708,0.336400,0.455113,0.312728,0.000000,0.445791,0.227142,0.372704,0.289871,0.143361,...,0.225374,0.276977,0.456527,0.349261,0.340082,0.397842,0.259906,0.189372,0.194699,0.426407


In [11]:
if df.isna().values.any():
    print("NaN values found in distance matrix!")

NaN values found in distance matrix!


In [12]:
# Find indices where df has NaN
nan_locs = np.where(df.isna())

# Collect sequence ID pairs
nan_pairs = [(df.index[i], df.columns[j]) for i, j in zip(*nan_locs)]

if nan_pairs:
    print("NaN distances found between:")
    for a, b in nan_pairs:
        print(f"{a}  <->  {b}")
else:
    print("No NaN distances in the matrix.")

NaN distances found between:
G000430525.part_NZ_AUME01000007.1  <->  G000430585.part_NZ_AUMH01000005.1
G000430585.part_NZ_AUMH01000005.1  <->  G000430525.part_NZ_AUME01000007.1


In [13]:
df_clean = df.dropna(axis=0, how="any").dropna(axis=1, how="any")

In [14]:
df = copy.deepcopy(df_clean)

In [15]:
df

,G000430525.part_NZ_AUME01000007.1,G000424365.part_NZ_AUJT01000008.1,G000336655.part_NZ_AOLZ01000076.1,G000163775.part_NZ_JH815302.1,G000430745.part_NZ_KE387231.1,G001894865.part_NZ_BCXA01000020.1,G900110905.part_FOFH01000009.1,G000300115.part_NZ_JH930378.1,G000172095.part_NZ_ABID01000003.1,G000430745.part_NZ_AUMP01000016.1,...,G900097105.part_NZ_LT629973.1,G001313205.part_BBFK01000002.1,G000336655.part_NZ_AOLZ01000022.1,G001005215.part_NZ_LBNQ01000037.1,G000497755.part_NZ_AYOD01000011.1,G000519205.part_NZ_JAFB01000009.1,G000224335.part_NZ_AFXZ01000019.1,G001308105.part_NZ_CP012851.1,G000425585.part_NZ_AUDS01000008.1,G000719275.part_NZ_JOFP01000004.1
G000424365.part_NZ_AUJT01000008.1,0.352161,0.000000,0.466374,0.458266,0.336400,0.426815,0.400139,0.441435,0.353202,0.332600,...,0.290732,0.446449,0.469780,0.368003,0.378607,0.406565,0.415452,0.298610,0.275142,0.419159
G000336655.part_NZ_AOLZ01000076.1,0.448177,0.466374,0.000000,0.541569,0.455113,0.163450,0.487712,0.536074,0.425270,0.445568,...,0.356746,0.457994,0.029615,0.452411,0.215755,0.191424,0.519376,0.538388,0.446742,0.188228
G000163775.part_NZ_JH815302.1,0.329393,0.458266,0.541569,0.000000,0.312728,0.497701,0.269718,0.315861,0.297955,0.320349,...,0.328771,0.287472,0.541356,0.327808,0.436320,0.460054,0.265488,0.281080,0.287741,0.466396
G000430745.part_NZ_KE387231.1,0.301708,0.336400,0.455113,0.312728,0.000000,0.445791,0.227142,0.372704,0.289871,0.143361,...,0.225374,0.276977,0.456527,0.349261,0.340082,0.397842,0.259906,0.189372,0.194699,0.426407
G001894865.part_NZ_BCXA01000020.1,0.402909,0.426815,0.163450,0.497701,0.445791,0.000000,0.465103,0.496512,0.318648,0.442533,...,0.315123,0.435424,0.156879,0.310014,0.179604,0.092480,0.477664,0.491535,0.384217,0.066311
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
G000519205.part_NZ_JAFB01000009.1,0.372078,0.406565,0.191424,0.460054,0.397842,0.092480,0.412835,0.470575,0.283864,0.404904,...,0.272981,0.384475,0.187616,0.273942,0.167213,0.000000,0.430847,0.436932,0.320229,0.067000
G000224335.part_NZ_AFXZ01000019.1,0.338929,0.415452,0.519376,0.265488,0.259906,0.477664,0.274715,0.305812,0.322953,0.282571,...,0.311589,0.297865,0.517336,0.339431,0.409110,0.430847,0.000000,0.232457,0.261883,0.450123
G001308105.part_NZ_CP012851.1,0.297073,0.298610,0.538388,0.281080,0.189372,0.491535,0.254758,0.302673,0.296036,0.205368,...,0.221865,0.279845,0.536382,0.345724,0.408130,0.436932,0.232457,0.000000,0.171358,0.461782
G000425585.part_NZ_AUDS01000008.1,0.286769,0.275142,0.446742,0.287741,0.194699,0.384217,0.210864,0.361583,0.226889,0.198694,...,0.180228,0.272130,0.449597,0.253664,0.310231,0.320229,0.261883,0.171358,0.000000,0.341977


In [166]:
# Optional: save to CSV/TSV
df.to_csv(os.path.join(path, "pairwise_distances_cafe_{}_phylip.tsv".format(dist_name.lower())), sep="\t")
#df.to_csv(os.path.join(path, "pairwise_distances_cafe_d2shepp_phylip.tsv"), sep="\t")

In [16]:
import pandas as pd
from itertools import combinations, product

# df is your pairwise distance matrix DataFrame
# Leaves labeled like "sample1.part_001", "sample2.part_010", etc.

# --- Step 1: group contigs by sample ID ---
sample_to_contigs = {}
for leaf in df.index:
    sample_id = leaf.split(".part")[0]  # extract sample ID
    sample_to_contigs.setdefault(sample_id, []).append(leaf)

# --- Step 2: compute within-sample distances ---
within_distances = []
records = []
for sample, contigs in sample_to_contigs.items():
    if len(contigs) < 2:
        continue  # no pairs to compute
    # all pairwise combinations of contigs in the same sample
    for a, b in combinations(contigs, 2):
        within_distances.append(df.loc[a, b])
        records.append({
            'Contig1': a,
            'Contig2': b,
            'Distance': df.loc[a, b],
            'Type': 'Within',
            'Sample1': sample,
            'Sample2': sample
        })

mean_within = sum(within_distances) / len(within_distances)
print("Mean within-sample distance:", mean_within)

# --- Step 3: compute across-sample distances ---
across_distances = []
samples = list(sample_to_contigs.keys())
for i in range(len(samples)):
    for j in range(i + 1, len(samples)):
        contigs_i = sample_to_contigs[samples[i]]
        contigs_j = sample_to_contigs[samples[j]]
        # all pairs between two different samples
        for a, b in product(contigs_i, contigs_j):
            across_distances.append(df.loc[a, b])
            records.append({
                'Contig1': a,
                'Contig2': b,
                'Distance': df.loc[a, b],
                'Type': 'Across',
                'Sample1': samples[i],
                'Sample2': samples[j]
            })

mean_across = sum(across_distances) / len(across_distances)
print("Mean across-sample distance:", mean_across)

# --- Step 4: save to text file ---
output_df = pd.DataFrame(records)

all_distances = output_df['Distance'].values
min_val = all_distances.min()
max_val = all_distances.max()
output_df['Distance_scaled'] = (output_df['Distance'] - min_val) / (max_val - min_val)
output_df['Condition'] ='cafe_phylip_{}'.format(dist_name.lower())


output_df.to_csv(os.path.join(path, "pairwise_distances_cafe_{}_phylip_R.txt".format(dist_name.lower())), sep='\t', index=False)
print("Saved distances")

Mean within-sample distance: 0.09779039766818837
Mean across-sample distance: 0.34508310608900855
Saved distances


In [56]:
0.345/0.097

3.556701030927835

In [9]:
max_val

nan

In [36]:
mean_across/mean_within

3.2316239322737905

In [37]:
# To scale distances relative to the combined range
import numpy as np

# Convert to numpy arrays if not already
within = np.array(within_distances)
across = np.array(across_distances)

# Combine to get the global min and max
all_distances = np.concatenate([within, across])
min_val = all_distances.min()
max_val = all_distances.max()

# Scale each set relative to the combined range
within_scaled = (within - min_val) / (max_val - min_val)
across_scaled = (across - min_val) / (max_val - min_val)

# Example: check min/max after scaling
print("Within scaled:", within_scaled.min(), "-", within_scaled.max())
print("Across scaled:", across_scaled.min(), "-", across_scaled.max())


Within scaled: 0.0 - 0.8217103006440996
Across scaled: 0.0 - 1.0


In [38]:
mean_scaled_within = sum(within_scaled) / len(within_scaled)
print("Mean within-sample distance:", mean_scaled_within)
mean_scaled_across = sum(across_scaled) / len(across_scaled)
print("Mean across-sample distance:", mean_scaled_across)

Mean within-sample distance: 0.03433655863462039
Mean across-sample distance: 0.1109628446347678


In [39]:
mean_scaled_across/mean_scaled_within

3.231623932250675

3.2316239322737905

In [28]:
from scipy.stats import f_oneway

# within_distances and across_distances are your lists of pairwise distances
F_stat, p_value = f_oneway(within_distances, across_distances)

print("F-statistic:", F_stat)
print("p-value:", p_value)

F-statistic: 7820.833961820503
p-value: 0.0


In [11]:
# Save to CSV/TSV
#matrix.to_csv("pairwise_distances.tsv", sep="\t")

#print("Distance matrix saved. Shape:", matrix.shape)

In [28]:
from functools import reduce

In [75]:
dist_emb=pd.read_csv('k7_v39_8k_s28_clade_All_TOL_Contigs_Global_logest/distances_kf2vec_emb_R.txt',sep="\t")
dist_pl=pd.read_csv('k7_v39_8k_s28_clade_All_TOL_Contigs_Global_logest/distances_kf2vec_placement_R.txt',sep="\t")
dist_fastme=pd.read_csv('fastme/distances_kf2vec_fastme_R.txt',sep="\t")

dist_d2star=pd.read_csv('cafe_dist/pairwise_distances_cafe_d2star_phylip_R.txt',sep="\t")
dist_d2shepp=pd.read_csv('cafe_dist/pairwise_distances_cafe_d2shepp_phylip_R.txt',sep="\t")
dist_cvtree=pd.read_csv('cafe_dist/pairwise_distances_cafe_cvtree_phylip_R.txt',sep="\t")
dist_cosine=pd.read_csv('cafe_dist/pairwise_distances_cafe_cosine_phylip_R.txt',sep="\t")

dist_cophylog=pd.read_csv('cafe_dist/pairwise_distances_cafe_co-phylog_phylip_R.txt',sep="\t")
dist_eu=pd.read_csv('cafe_dist/pairwise_distances_cafe_eu_phylip_R.txt',sep="\t")
dist_js=pd.read_csv('cafe_dist/pairwise_distances_cafe_js_phylip_R.txt',sep="\t")
dist_ma=pd.read_csv('cafe_dist/pairwise_distances_cafe_ma_phylip_R.txt',sep="\t")




In [76]:
dfs = [dist_emb, dist_ma, dist_js, dist_eu, dist_cophylog, dist_cosine, dist_cvtree, dist_d2shepp, dist_d2star, dist_fastme, dist_pl]

In [77]:
# Step 1: normalize pairs in all DataFrames
for df in dfs:
    df["pair"] = df[["Contig1", "Contig2"]].apply(lambda x: tuple(sorted(x)), axis=1)

In [78]:
# Step 2: find pairs common to all DataFrames
common_pairs = set(dfs[0]["pair"])
for df in dfs[1:]:
    common_pairs &= set(df["pair"])

In [79]:
# Step 3: filter each DataFrame to keep only common pairs
filtered_dfs = [df[df["pair"].isin(common_pairs)] for df in dfs]


In [81]:
# Step 4: concatenate vertically (optional)
result = pd.concat(filtered_dfs, ignore_index=True).drop(columns=["pair"])

In [82]:
result.to_csv(os.path.join(path, "pairwise_distances_kf2vec_cafe_R.txt".format(dist_name.lower())), sep='\t', index=False)

In [80]:
len(common_pairs)

2552670

In [83]:
dfs_names = [
    "dist_emb", "dist_ma", "dist_js", "dist_eu", "dist_cophylog",
    "dist_cosine", "dist_cvtree", "dist_d2shepp", "dist_d2star",
    "dist_fastme", "dist_pl"
]

for df, name in zip(filtered_dfs, dfs_names):
    df.to_csv(f"{name}_filtered.csv", index=False, sep='\t', )